# Experiment 04: Term-Frequency, NER, and TF-IDF Analysis

- **Course:** Computational Linguistics and Natural Language Processing (CLNLP) Lab  
- **Date:** 2026-08-27  
- **Student Name:** Mradul Lakhotiya  
- **Roll Number:** [Roll Number]  

---

## 1. Aim & Objectives
- **Aim:** To compute term frequencies, extract Named Entities using toolkits, and perform a complete TF-IDF calculation from scratch to analyze term importance across documents.
- **Objectives:**
  1. Perform Term-Frequency Analysis and Named Entity Recognition (NER) on a given text using a toolkit.
  2. Perform Term-Frequency Analysis on a given text without using any NLP toolkit.
  3. Calculate TF, DF, IDF, and TF-IDF without any NLP toolkit for multiple documents and identify top terms.

## 2. Environment Setup & Imports

In [1]:
import os
import csv
import math
from collections import Counter
import string

import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
import spacy

# Download nltk dependencies if needed
nltk.download('punkt')

try:
    nlp = spacy.load('en_core_web_sm')
except:
    import spacy.cli
    spacy.cli.download('en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

# Create outputs directory
os.makedirs('outputs', exist_ok=True)

print("Setup Complete.")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mradu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Setup Complete.


## 3. Dataset / Corpus Input

In [2]:
# Load the primary text for Exp 4.1 and 4.2
with open('data/input_text.txt', 'r', encoding='utf-8') as f:
    main_text = f.read()

print(f"Loaded main text of length: {len(main_text)} characters.")

# Three documents for Exp 4.3
documents = [
    "Natural language processing is a field of artificial intelligence.",
    "Natural language processing helps computers understand human language.",
    "Machine learning is an important part of artificial intelligence."
]


Loaded main text of length: 17345 characters.


## 4. Core Implementation & Results

### Experiment 4.1: TF & NER (Using Toolkit)

In [3]:
print("--- Named Entity Recognition (NER) ---")
doc_spacy = nlp(main_text)
entities = set([(ent.text, ent.label_) for ent in doc_spacy.ents])
for ent, label in list(entities)[:10]:  # display a few entities
    print(f"{ent} -> {label}")

print("\n--- Term Frequency (TF) ---")
# Tokenize and lowercase
tokens = [word.lower() for word in word_tokenize(main_text) if word.isalnum()]
tf_counter = Counter(tokens)

# Save to CSV using pandas
df_tf = pd.DataFrame(tf_counter.items(), columns=['Term', 'Frequency'])
df_tf = df_tf.sort_values(by='Frequency', ascending=False)
df_tf.to_csv('outputs/4_1_tf.csv', index=False)

print("Top 10 Most Frequent Terms:")
display(df_tf.head(10))

--- Named Entity Recognition (NER) ---
first -> ORDINAL
Natural Language Processing and Artificial Intelligence

Natural Language Processing -> ORG
thousands -> CARDINAL
Retrieval-Augmented Generation -> ORG
hundreds or thousands -> CARDINAL
Chatbots -> PERSON
Python -> GPE
Bengali -> NORP
NLP -> ORG
English -> LANGUAGE

--- Term Frequency (TF) ---
Top 10 Most Frequent Terms:


,Term,Frequency
96,a,88
3,and,86
50,the,75
20,to,42
42,in,42
99,can,42
14,of,40
10,is,37
157,word,30
111,may,30


### Experiment 4.2: TF (Without Toolkit)

In [4]:
# Clean and tokenize manually
cleaned_text = main_text.lower().translate(str.maketrans('', '', string.punctuation))
manual_tokens = cleaned_text.split()

manual_tf = {}
for term in manual_tokens:
    manual_tf[term] = manual_tf.get(term, 0) + 1

# Sort dict
sorted_tf = sorted(manual_tf.items(), key=lambda item: item[1], reverse=True)

# Save to CSV
with open('outputs/4_2_tf.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Term', 'Frequency'])
    writer.writerows(sorted_tf)

print("Top 10 Most Frequent Terms (Manual Method):")
for term, freq in sorted_tf[:10]:
    print(f"{term}: {freq}")

Top 10 Most Frequent Terms (Manual Method):
a: 90
and: 86
the: 75
to: 42
in: 42
can: 41
of: 40
is: 37
may: 30
word: 30


### Experiment 4.3: TF-IDF Without Toolkit

In [5]:
import math

# 1. Preprocessing
processed_docs = []
for doc in documents:
    doc_cleaned = doc.lower().translate(str.maketrans('', '', string.punctuation))
    processed_docs.append(doc_cleaned.split())

# 2. Calculate TF for every term
tfs = []
for tokens in processed_docs:
    tf_dict = {}
    total_terms = len(tokens)
    for term in tokens:
        tf_dict[term] = tf_dict.get(term, 0) + 1
    # Convert to standard TF representation (count / total terms)
    # For simplicity based on prompt, we will just use raw count.
    # Using raw count as TF is common in introductory NLP.
    tfs.append(tf_dict)

# 3. Calculate DF for every term
df_dict = {}
for tf_dict in tfs:
    for term in tf_dict.keys():
        df_dict[term] = df_dict.get(term, 0) + 1

# 4. Calculate IDF using logarithmic formula (log10)
N = len(documents)
idf_dict = {}
for term, df in df_dict.items():
    idf_dict[term] = math.log10(N / df)

# 5. Calculate TF-IDF & Display all metrics
for i, tf_dict in enumerate(tfs):
    print(f"\n{'='*40}\nDocument {i+1}: {documents[i]}\n{'='*40}")
    tfidf_scores = []
    
    print(f"{'Term':<15} | {'TF':<5} | {'DF':<5} | {'IDF':<8} | {'TF-IDF':<8}")
    print("-"*50)
    
    for term, tf in tf_dict.items():
        df = df_dict[term]
        idf = idf_dict[term]
        tfidf = tf * idf
        tfidf_scores.append((term, tf, df, idf, tfidf))
        print(f"{term:<15} | {tf:<5} | {df:<5} | {idf:<8.4f} | {tfidf:<8.4f}")
        
    # 6. Identify top 10 terms with highest TF-IDF
    tfidf_scores.sort(key=lambda x: x[4], reverse=True)
    print("\nTop Terms (TF-IDF):")
    for term, _, _, _, tfidf in tfidf_scores[:10]:
        print(f" - {term}: {tfidf:.4f}")


Document 1: Natural language processing is a field of artificial intelligence.
Term            | TF    | DF    | IDF      | TF-IDF  
--------------------------------------------------
natural         | 1     | 2     | 0.1761   | 0.1761  
language        | 1     | 2     | 0.1761   | 0.1761  
processing      | 1     | 2     | 0.1761   | 0.1761  
is              | 1     | 2     | 0.1761   | 0.1761  
a               | 1     | 1     | 0.4771   | 0.4771  
field           | 1     | 1     | 0.4771   | 0.4771  
of              | 1     | 2     | 0.1761   | 0.1761  
artificial      | 1     | 2     | 0.1761   | 0.1761  
intelligence    | 1     | 2     | 0.1761   | 0.1761  

Top Terms (TF-IDF):
 - a: 0.4771
 - field: 0.4771
 - natural: 0.1761
 - language: 0.1761
 - processing: 0.1761
 - is: 0.1761
 - of: 0.1761
 - artificial: 0.1761
 - intelligence: 0.1761

Document 2: Natural language processing helps computers understand human language.
Term            | TF    | DF    | IDF      | TF-IDF  
-----

## 5. Conclusion & Key Findings
- **Summary:** We computed Term Frequency using toolkits and manual string methods, finding consistent results. Named Entity Recognition correctly extracted conceptual entities from the text. The TF-IDF analysis demonstrated how common words (like 'is' or 'of') receive a low IDF weight, effectively filtering them out, whereas unique words get heavily weighted.
- **Linguistic Insights:** Removing punctuation and converting to lowercase is paramount to ensure terms like 'Intelligence.' and 'intelligence' map to the same token.
- **Limitations & Future Scope:** In our manual TF-IDF implementation, words that appear in all documents get an IDF of $0$ (since $\log_{10}(3/3) = 0$), completely neutralizing their TF-IDF. This highlights why TF-IDF is exceptional for finding distinctive, keyword-like features in documents.